In [1]:
import os
import sys
from dotenv import load_dotenv

load_dotenv()

sys.path.append(os.environ["GPTNL_DATALOADER"])

EPOCH_1_FOLDERS = {
    "train": os.path.join(os.environ["GPTNL_1_TOKENIZED"], "buckets_split_train"),
    "validate": os.path.join(os.environ["GPTNL_1_TOKENIZED"], "buckets_split_validate"),
}

EPOCH_2_FOLDERS = {
    "train": os.path.join(os.environ["GPTNL_2_TOKENIZED"], "buckets_split_train"),
    "validate": os.path.join(os.environ["GPTNL_2_TOKENIZED"], "buckets_split_validate"),
}

In [2]:
from sampling.functions import count_tokens
import copy


def get_bucket_from_fp(file_path):
    if "code" in file_path:
        return "code"
    elif "dutch_high_quality" in file_path:
        return "dutch_high_quality"
    elif "dutch_low_medium_quality" in file_path:
        return "dutch_low_medium_quality"
    elif "english_high_quality" in file_path:
        return "english_high_quality"
    elif "english_low_medium_quality" in file_path:
        return "english_low_medium_quality"
    elif "new_high_quality" in file_path:
        return "new_high_quality"
    elif "new_low_medium_quality" in file_path:
        return "new_low_medium_quality"
    elif "other_languages" in file_path:
        return "other_languages"
    else:
        raise ValueError(f"Unknown bucket for file path: {file_path}")


def count_tokens_in_folders(folders):
    counts_per_bucket = {
        "total": 0,
        "code": 0,
        "dutch_high_quality": 0,
        "dutch_low_medium_quality": 0,
        "english_high_quality": 0,
        "english_low_medium_quality": 0,
        "new_high_quality": 0,
        "new_low_medium_quality": 0,
        "other_languages": 0,
    }
    counts_per_split = {
        split: copy.deepcopy(counts_per_bucket) for split in folders.keys()
    }
    for split, folder in folders.items():
        if os.path.isdir(folder):
            print(f"Counting tokens in {split} set...")
            count = counts_per_bucket
            for root, dirs, files in os.walk(folder):
                for file in files:
                    if file.endswith(".npy"):
                        file_path = os.path.join(root, file)
                        bucket = get_bucket_from_fp(file_path)
                        # print(f"Processing {file_path}...")
                        count["total"] += count_tokens(file_path)
                        counts_per_bucket[bucket] += count_tokens(file_path)
                        counts_per_split[split]["total"] += count_tokens(file_path)
                        counts_per_split[split][bucket] += count_tokens(file_path)
    return counts_per_bucket, counts_per_split

In [4]:
epoch_1_counts, epoch_1_split_counts = count_tokens_in_folders(EPOCH_1_FOLDERS)
epoch_2_counts, epoch_2_split_counts = count_tokens_in_folders(EPOCH_2_FOLDERS)

Counting tokens in train set...
Counting tokens in validate set...
Counting tokens in train set...
Counting tokens in validate set...


In [5]:
import pandas as pd

# Create a comprehensive dataframe with all counts
data = []

# Add epoch 1 data
for split, counts in epoch_1_split_counts.items():
    for bucket, count in counts.items():
        data.append(
            {"epoch": "epoch_1", "split": split, "bucket": bucket, "token_count": count}
        )

# Add epoch 2 data
for split, counts in epoch_2_split_counts.items():
    for bucket, count in counts.items():
        data.append(
            {"epoch": "epoch_2", "split": split, "bucket": bucket, "token_count": count}
        )

# Create DataFrame
df_overview = pd.DataFrame(data)

# Create a pivot table for better readability
df_pivot = df_overview.pivot_table(
    index=["epoch", "split"], columns="bucket", values="token_count", fill_value=0
)

print("Token counts overview:")
df_pivot

Token counts overview:


bucket                    code  dutch_high_quality  dutch_low_medium_quality  \
epoch   split                                                                  
epoch_1 train     8.393435e+10        1.458467e+10              6.133430e+09   
        validate  1.704255e+09        2.914704e+08              1.264540e+08   
epoch_2 train     2.273092e+11        2.088203e+10              6.133568e+09   
        validate  4.640329e+09        4.187061e+08              1.255695e+08   

bucket            english_high_quality  english_low_medium_quality  \
epoch   split                                                        
epoch_1 train             4.853224e+10                1.552211e+11   
        validate          9.864046e+08                3.169007e+09   
epoch_2 train             4.853165e+10                1.552223e+11   
        validate          9.843609e+08                3.167814e+09   

bucket            new_high_quality  new_low_medium_quality  other_languages  \
epoch   split                                                                 
epoch_1 train         0.000000e+00            0.000000e+00     4.736504e+10   
        validate      0.000000e+00            0.000000e+00     9.652494e+08   
epoch_2 train         2.444355e+10            8.331059e+09     4.737296e+10   
        validate      4.991515e+08            1.699187e+08     9.667039e+08   

bucket                   total  
epoch   split                   
epoch_1 train     3.557708e+11  
        validate  7.242841e+09  
epoch_2 train     5.382263e+11  
        validate  1.097255e+10

In [6]:
# Create a copy of the pivot table for formatting
df_formatted = df_pivot.copy()

# Convert to billions and round to 2 decimal places
df_formatted = df_formatted / 1e9
df_formatted = df_formatted.round(2)

# Add total rows for each epoch (train + validate)
epochs = df_formatted.index.get_level_values("epoch").unique()
for epoch in epochs:
    epoch_data = df_formatted.loc[epoch]
    total_row = epoch_data.sum()
    # Add the total row to the dataframe
    df_formatted.loc[(epoch, "total"), :] = total_row

# Rename columns to nice looking names
column_mapping = {
    "code": "Code",
    "dutch_high_quality": "Dutch (high quality)",
    "dutch_low_medium_quality": "Dutch (low/medium quality)",
    "english_high_quality": "English (high quality)",
    "english_low_medium_quality": "English (low/medium quality)",
    "new_high_quality": "New (high quality)",
    "new_low_medium_quality": "New (low/medium quality)",
    "other_languages": "Other languages",
    "total": "Total",
}

df_formatted = df_formatted.rename(columns=column_mapping)

print("Token counts (in billions):")
df_formatted = df_formatted.transpose()

df_formatted

Token counts (in billions):


epoch                        epoch_1          epoch_2          epoch_1 epoch_2
split                          train validate   train validate   total   total
bucket                                                                        
Code                           83.93     1.70  227.31     4.64   85.63  231.95
Dutch (high quality)           14.58     0.29   20.88     0.42   14.87   21.30
Dutch (low/medium quality)      6.13     0.13    6.13     0.13    6.26    6.26
English (high quality)         48.53     0.99   48.53     0.98   49.52   49.51
English (low/medium quality)  155.22     3.17  155.22     3.17  158.39  158.39
New (high quality)              0.00     0.00   24.44     0.50    0.00   24.94
New (low/medium quality)        0.00     0.00    8.33     0.17    0.00    8.50
Other languages                47.37     0.97   47.37     0.97   48.34   48.34
Total                         355.77     7.24  538.23    10.97  363.01  549.20

In [7]:
# Create a simplified comparison table with only total counts for each epoch
df_comparison = df_formatted[[("epoch_1", "total"), ("epoch_2", "total")]].copy()
df_comparison.columns = ["Epoch 1 Total", "Epoch 2 Total"]

# Remove the 'Total' row to avoid redundancy
df_comparison = df_comparison.drop("Total")

# Calculate percentages for each epoch
epoch_1_total = df_comparison["Epoch 1 Total"].sum()
epoch_2_total = df_comparison["Epoch 2 Total"].sum()

df_comparison["Epoch 1 %"] = (
    df_comparison["Epoch 1 Total"] / epoch_1_total * 100
).round(2)
df_comparison["Epoch 2 %"] = (
    df_comparison["Epoch 2 Total"] / epoch_2_total * 100
).round(2)

# Export to markdown
comparison_markdown = df_comparison.to_markdown()

# Save to file
with open("token_counts.md", "w") as f:
    f.write("# Token Count Comparison: Epoch 1 vs Epoch 2\n\n")
    f.write("Token counts in billions for each data bucket with percentages:\n\n")
    f.write(comparison_markdown)

print("Epoch comparison table with percentages:")
print(df_comparison)
print("\nMarkdown saved to 'token_counts.md'")

Epoch comparison table with percentages:
                              Epoch 1 Total  Epoch 2 Total  Epoch 1 %  \
bucket                                                                  
Code                                  85.63         231.95      23.59   
Dutch (high quality)                  14.87          21.30       4.10   
Dutch (low/medium quality)             6.26           6.26       1.72   
English (high quality)                49.52          49.51      13.64   
English (low/medium quality)         158.39         158.39      43.63   
New (high quality)                     0.00          24.94       0.00   
New (low/medium quality)               0.00           8.50       0.00   
Other languages                       48.34          48.34      13.32   

                              Epoch 2 %  
bucket                                   
Code                              42.23  
Dutch (high quality)               3.88  
Dutch (low/medium quality)         1.14  
English (high qual

# Count tokens per dataset (estimated)

In [8]:
import numpy as np
from pyarrow import parquet as pq
import pyarrow.compute as pc
import pyarrow as pa
from collections import defaultdict
import json
import logging

# Configure logging
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)


EPOCH_2_FOLDERS = {
    "train": os.path.join(os.environ["GPTNL_2_TOKENIZED"], "buckets_split_train"),
    "validate": os.path.join(os.environ["GPTNL_2_TOKENIZED"], "buckets_split_validate"),
}

UNTOKENIZED_EPOCH_2_FOLDERS = {
    "train": os.path.join(
        os.environ["GPTNL_SCRATCH"], "gpt_nl_dataset_v2.0_bucketed/buckets_split_train"
    ),
    "validate": os.path.join(
        os.environ["GPTNL_SCRATCH"],
        "gpt_nl_dataset_v2.0_bucketed/buckets_split_validate",
    ),
}

buckets = [
    "code",
    "dutch_high_quality",
    "dutch_low_medium_quality",
    "english_high_quality",
    "english_low_medium_quality",
    "new_high_quality",
    "new_low_medium_quality",
    "other_languages",
]

# Restructure to track progress: {train_valid: {bucket: {dataset_source: count}}}
counts_per_dataset_source = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))


def load_counts(filename="counts_per_dataset_source.json"):
    """Load counts dictionary from JSON file if it exists"""
    save_path = filename

    if os.path.exists(save_path):
        with open(save_path, "r") as f:
            regular_dict = json.load(f)
        logger.info(f"Loaded existing counts from {save_path}")
        # Reconstruct nested defaultdicts
        result = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))
        for train_valid, buckets_data in regular_dict.items():
            for bucket, dataset_counts in buckets_data.items():
                for dataset_source, count in dataset_counts.items():
                    result[train_valid][bucket][dataset_source] = count
        return result
    return defaultdict(lambda: defaultdict(lambda: defaultdict(int)))


def get_completed_buckets(counts_dict, train_valid):
    """Get list of buckets that have already been processed for a given train_valid split"""
    return list(counts_dict.get(train_valid, {}).keys())


def get_doc_lengths(tokens, eos_token_id: int) -> list[int]:
    eos_positions = np.where(tokens == eos_token_id)[0]
    if len(eos_positions) == 0:
        return [len(tokens)]

    doc_lengths = [eos_positions[0] + 1]  # +1 to include the EOS token itself

    if len(eos_positions) > 1:
        doc_lengths.extend(np.diff(eos_positions).tolist())

    if eos_positions[-1] < len(tokens) - 1:
        remaining_length = len(tokens) - eos_positions[-1] - 1
        doc_lengths.append(remaining_length)

    return doc_lengths


def save_counts(counts_dict, filename="counts_per_dataset_source.json"):
    """Save counts dictionary to JSON file"""
    save_path = filename
    # Convert nested defaultdicts to regular dicts for JSON serialization
    regular_dict = {}
    for train_valid, buckets_data in counts_dict.items():
        regular_dict[train_valid] = {}
        for bucket, dataset_counts in buckets_data.items():
            regular_dict[train_valid][bucket] = dict(dataset_counts)

    with open(save_path, "w") as f:
        json.dump(regular_dict, f, indent=2)
    logger.info(f"Saved counts to {save_path}")


# Load existing counts if available
counts_per_dataset_source = load_counts("counts_intermediate.json")

for train_valid, tokenized_folder, untokenized_folder in zip(
    ["train", "validate"],
    EPOCH_2_FOLDERS.values(),
    UNTOKENIZED_EPOCH_2_FOLDERS.values(),
):
    logger.info(f"Processing {train_valid}...")
    logger.info(f"  Tokenized folder: {tokenized_folder}")
    logger.info(f"  Untokenized folder: {untokenized_folder}")

    # Check which buckets are already completed
    completed_buckets = get_completed_buckets(counts_per_dataset_source, train_valid)
    logger.info(f"Already completed buckets for {train_valid}: {completed_buckets}")

    for bucket in reversed(buckets):
        if bucket == "code":
            continue

        # Skip if this bucket is already completed
        if bucket in completed_buckets:
            logger.info(f"Skipping already completed bucket: {bucket}")
            continue

        logger.info(f"Processing bucket: {bucket}")

        tokenized_path = os.path.join(tokenized_folder, bucket)
        untokenized_path = os.path.join(untokenized_folder, bucket)

        for i, npy_file in enumerate(os.listdir(tokenized_path)):
            pqt_file = npy_file.replace(".npy", ".parquet").replace("_tokenized", "")

            # print(pqt_file.split("/")[-1].replace(".parquet", ""))
            print("")

            pqt_fp = os.path.join(untokenized_path, pqt_file)
            npy_fp = os.path.join(tokenized_path, npy_file)

            if not os.path.exists(pqt_fp):
                logger.warning(
                    f"  {pqt_file} does not exist in {untokenized_path}, but {npy_fp} exists in {tokenized_path}."
                )

            array = np.memmap(npy_fp, dtype=np.uint32)
            n_tokens = array.shape[0]
            doc_lengths = get_doc_lengths(
                array, eos_token_id=2
            )  # Assuming EOS token ID is 2

            # Read parquet metadata to get number of rows
            parquet_file = pq.ParquetFile(pqt_fp)
            num_rows_parquet = parquet_file.metadata.num_rows
            sub_pqt = parquet_file.read(
                ["n_non_symbol_words", "dataset_name", "source"]
            )

            # 2. Group by dataset_name + source combination efficiently
            # Create a combined column for grouping
            dataset_source = pc.binary_join_element_wise(
                sub_pqt["dataset_name"], sub_pqt["source"], "---"
            )
            sub_pqt = sub_pqt.add_column(0, "dataset_source", dataset_source)

            # We can infer the tokens exactly by using the document lengths
            if len(doc_lengths) == num_rows_parquet:
                sub_pqt = sub_pqt.add_column(
                    0, "token_count", pa.array(doc_lengths, type=pa.int64())
                )
                grouped = sub_pqt.select(["token_count", "dataset_source"])
                result = grouped.group_by("dataset_source").aggregate(
                    [("token_count", "sum")]
                )

                for i in range(len(result)):
                    combo = result["dataset_source"][i].as_py()
                    token_count = result["token_count_sum"][i].as_py()
                    counts_per_dataset_source[train_valid][bucket][combo] += token_count

            # Else: We need to make an estimate based on the fraction of n_non_symbol_words or n_rows per dataset_name + source
            else:
                # 1. Total number of n_non_symbol_words (vectorized sum)
                total_words = pc.sum(sub_pqt["n_non_symbol_words"]).as_py()

                # If we have this information use the word fraction (word_count/total_words) to attribute tokens to dataset_name + source groups
                if total_words:
                    grouped = sub_pqt.select(["n_non_symbol_words", "dataset_source"])
                    result = grouped.group_by("dataset_source").aggregate(
                        [("n_non_symbol_words", "sum")]
                    )

                    # Calculate fractions based on n_non_symbol_words
                    for i in range(len(result)):
                        combo = result["dataset_source"][i].as_py()
                        word_count = result["n_non_symbol_words_sum"][i].as_py()
                        fraction = word_count / total_words
                        counts_per_dataset_source[train_valid][bucket][combo] += (
                            fraction * n_tokens
                        )

                # If we don't have word counts, we can only estimate based on the number of rows
                else:
                    logger.warning(
                        "Last resort, using row counts to estimate token counts."
                    )
                    # Calculate fractions based on number of rows since total_words is 0
                    result = sub_pqt.group_by("dataset_source").aggregate(
                        [("dataset_source", "count")]
                    )

                    for i in range(len(result)):
                        combo = result["dataset_source"][i].as_py()
                        row_count = result["dataset_source_count"][i].as_py()
                        fraction = row_count / num_rows_parquet
                        counts_per_dataset_source[train_valid][bucket][combo] += (
                            fraction * n_tokens
                        )

        # Save intermediate progress after each bucket
        save_counts(counts_per_dataset_source, "counts_intermediate.json")
        print(counts_per_dataset_source[train_valid][bucket])
        logger.info(f"Completed bucket {bucket} for {train_valid}")

# Save final counts after processing everything
save_counts(counts_per_dataset_source, "counts_final_all.json")

2025-08-07 15:42:38,843 - INFO - Processing train...
2025-08-07 15:42:38,845 - INFO -   Tokenized folder: GPTNL_2_TOKENIZED/buckets_split_train
2025-08-07 15:42:38,846 - INFO -   Untokenized folder: GPTNL_SCRATCH/gpt_nl_dataset_v2.0_bucketed/buckets_split_train
2025-08-07 15:42:38,846 - INFO - Already completed buckets for train: []
2025-08-07 15:42:38,847 - INFO - Processing bucket: other_languages


In [5]:
with open(
    os.path.join(os.environ["GPTNL_DATALOADER"], "count_tokens/counts_final_all.json"),
    "r",
) as fp:
    counts = json.load(fp)

print(counts)